In [56]:
import json

with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [60]:
for course_dict in docs_raw:
    print(course_dict["course"])


data-engineering-zoomcamp
machine-learning-zoomcamp
mlops-zoomcamp


In [3]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

documents[1]

{'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites',
 'section': 'General course-related questions',
 'question': 'Course - What are the prerequisites for this course?',
 'course': 'data-engineering-zoomcamp'}

In [5]:
# Create embeddings using pretrained model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

In [11]:
len(model.encode("This is a simple sentence"))

768

In [12]:
# create dense vector using the pre-trained model
operations = []
for doc in documents:
    # transforming the title into an embedding using the model
    doc['text_vector'] = model.encode(doc['text']).tolist()
    operations.append(doc)

In [70]:
operations[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp',
 'text_vector': [-0.057206593453884125,
  0.019887374714016914,
  -0.024336032569408417,
  -0.010076598264276981,
  0.03898910805583,
  0.010869299061596394,
  0.02646889165043831,
  -0.0339190736413002,
  0.029106372967362404,
  -0.04912998527288437,
  0.033005863428115845,
  -0.06733977794647217,
  -0.02142668515443802,
  -0.01319753099232912,
  -0.03563873469829559,
  0.06667864322662354,
  -0.0419478714466095,
  -0.05011487007141113,
  -0.05269116908311844,
  -0.047963038086891174,
  -0.06410781294107437,
  -0.0028072858694940805,
  -0.04610515758395195,
  0.02531910128891468,
  -0.006165231578052044,
 

In [13]:
# Set up Elastic search connection
from elasticsearch import Elasticsearch

es_client = Elasticsearch('http://localhost:9200')
es_client.info()

ObjectApiResponse({'name': 'f3abc204ec4d', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'ybfCF3UGRtKrSeZ6wUkmyg', 'version': {'number': '8.11.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6f9ff581fbcde658e6f69d6ce03050f060d1fd0c', 'build_date': '2023-11-11T10:05:59.421038163Z', 'build_snapshot': False, 'lucene_version': '9.8.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [14]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "text"},
            "text_vector": {"type": "dense_vector", "dims": 768, "index": True, "similarity": "cosine"},
        }
    }
}

In [16]:
index_name = "course-questions"

es_client.indices.delete(index=index_name, ignore_unavailable=True)
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [71]:
for doc in operations:
    try:
        es_client.index(index=index_name, document=doc)
    except Exception as e:
        print(e)

### Step 6: Create end user query

In [72]:
search_term = "windows or mac?"
vector_search_term = model.encode(search_term)

In [73]:
len(vector_search_term)

768

In [74]:
query = {
    "field": "text_vector",
    "query_vector": vector_search_term,
    "k": 5,
    "num_candidates": 10000
}

In [75]:
response = es_client.search(index=index_name,
                            knn=query, # approximate kNN search to run
                            source=["text", "section", "question", "course"] # which fields are to be returned for matched documents
                            ) 

In [76]:
response['hits']['hits']

[{'_index': 'course-questions',
  '_id': '_H92L5QBMwTH6tSPawnw',
  '_score': 0.714792,
  '_source': {'text': 'Yes! Linux is ideal but technically it should not matter. Students last year used all 3 OSes successfully',
   'section': 'General course-related questions',
   'question': 'Environment - Is the course [Windows/mac/Linux/...] friendly?',
   'course': 'data-engineering-zoomcamp'}},
 {'_index': 'course-questions',
  '_id': 'sH8KMJQBMwTH6tSPfQ0M',
  '_score': 0.714792,
  '_source': {'text': 'Yes! Linux is ideal but technically it should not matter. Students last year used all 3 OSes successfully',
   'section': 'General course-related questions',
   'question': 'Environment - Is the course [Windows/mac/Linux/...] friendly?',
   'course': 'data-engineering-zoomcamp'}},
 {'_index': 'course-questions',
  '_id': 'D392L5QBMwTH6tSPeg26',
  '_score': 0.6134738,
  '_source': {'text': 'If you wish to use WSL on your windows machine, here are the setup instructions:\nCommand: Sudo apt insta

#### Step 7: Perform semantic search & advanced search

In [77]:
response = es_client.search(
    index=index_name,
    query={
        "bool": {
            "must": {
                "multi_match": {"query": "windows or python?",
                                "fields": ["text", "question", "course", "title"],
                                "type": "best_fields"}
                                },
            "filter": {
                "term": {
                    "course": 'data-engineering-zoomcamp'
                    }
            }

        }
    }
)

In [78]:
response["hits"]

{'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}

In [84]:
knn_query = {
    "field": "text_vector",
    "query_vector": vector_search_term,
    "k": 5,
    "num_candidates": 10000
}

response = es_client.search(
    index=index_name,
    query={
        "match": {
            "course": "data-engineering-zoomcamp"
        },
    },
    knn=knn_query,
    size=5,
    explain=True
)

In [85]:
response["hits"]["hits"]

[{'_shard': '[course-questions][0]',
  '_node': 'C15KYXjjTFOlLBzcK8mTZg',
  '_index': 'course-questions',
  '_id': '_H92L5QBMwTH6tSPawnw',
  '_score': 2.2411344,
  '_source': {'text': 'Yes! Linux is ideal but technically it should not matter. Students last year used all 3 OSes successfully',
   'section': 'General course-related questions',
   'question': 'Environment - Is the course [Windows/mac/Linux/...] friendly?',
   'course': 'data-engineering-zoomcamp',
   'text_vector': [-0.026965469121932983,
    -0.0006259690853767097,
    -0.016629505902528763,
    0.052851416170597076,
    0.054765306413173676,
    -0.031339794397354126,
    0.029942618682980537,
    -0.04808564856648445,
    0.04467552527785301,
    0.005839465651661158,
    0.016233116388320923,
    0.012001181952655315,
    -0.0312223881483078,
    0.01660069078207016,
    -0.04886896535754204,
    -0.06496300548315048,
    0.04643423110246658,
    -0.009297611191868782,
    -0.06425280123949051,
    -0.01373269688338041